In [1]:
from pathlib import Path
import pandas as pd
import re

# Check the project output folder
possible_paths = [
    Path("../outputs/hdfs_incident_intelligence.csv"),
    Path("outputs/hdfs_incident_intelligence.csv")
]

incident_path = next(
    (path for path in possible_paths if path.exists()),
    None
)

if incident_path is None:
    raise FileNotFoundError(
        "Could not find hdfs_incident_intelligence.csv"
    )

# Load incident data
incidents_df = pd.read_csv(incident_path)

print("File loaded successfully!")
print("File path:", incident_path.resolve())
print("Dataset shape:", incidents_df.shape)
print("\nColumns:")
print(incidents_df.columns.tolist())

display(incidents_df.head())

File loaded successfully!
File path: D:\Srinath Folder\HDFS_Log_Anomaly_Detection_Project_Structure\HDFS_Log_Anomaly_Detection\outputs\hdfs_incident_intelligence.csv
Dataset shape: (17635, 7)

Columns:
['BlockId', 'ReconstructionScore', 'PredictedAnomaly', 'Severity', 'SequenceLength', 'TopEvents', 'TopEventDescriptions']


,BlockId,ReconstructionScore,PredictedAnomaly,Severity,SequenceLength,TopEvents,TopEventDescriptions
0,blk_5643631875592041420,1.383462,1,Medium,24,E5 (3); E11 (3); E9 (3); E26 (3); E23 (3),E5 (3): [*]Receiving block[*]src:[*]dest:[*] |...
1,blk_-7182584873419195739,0.917560,1,Low,37,E4 (17); E5 (3); E11 (3); E9 (3); E26 (3),E4 (17): [*]Got exception while serving[*]to[*...
2,blk_8765601758654707054,1.084821,1,Low,23,E5 (3); E11 (3); E9 (3); E26 (3); E23 (3),E5 (3): [*]Receiving block[*]src:[*]dest:[*] |...
3,blk_-6710278520007599935,1.294334,1,Medium,24,E26 (4); E5 (3); E11 (3); E9 (3); E23 (3),E26 (4): [*]BLOCK* NameSystem[*]addStoredBlock...
4,blk_3204685320797844679,2.709779,1,Medium,2,E5 (1); E22 (1),E5 (1): [*]Receiving block[*]src:[*]dest:[*] |...


In [2]:
# Extract unique Event IDs from each incident

events_df = incidents_df[
    ["BlockId", "Severity", "TopEvents"]
].copy()

events_df["EventId"] = events_df["TopEvents"].fillna("").apply(
    lambda text: list(dict.fromkeys(
        re.findall(r"\b(E\d+)\s*\(", text)
    ))
)

# Expand the event lists into individual rows
events_exploded = events_df.explode("EventId")

# Remove rows without an Event ID
events_exploded = events_exploded.dropna(subset=["EventId"])

print("Original incident records:", len(incidents_df))
print("Expanded event records:", len(events_exploded))

display(events_exploded.head(10))

Original incident records: 17635
Expanded event records: 72863


,BlockId,Severity,TopEvents,EventId
0,blk_5643631875592041420,Medium,E5 (3); E11 (3); E9 (3); E26 (3); E23 (3),E5
0,blk_5643631875592041420,Medium,E5 (3); E11 (3); E9 (3); E26 (3); E23 (3),E11
0,blk_5643631875592041420,Medium,E5 (3); E11 (3); E9 (3); E26 (3); E23 (3),E9
0,blk_5643631875592041420,Medium,E5 (3); E11 (3); E9 (3); E26 (3); E23 (3),E26
0,blk_5643631875592041420,Medium,E5 (3); E11 (3); E9 (3); E26 (3); E23 (3),E23
1,blk_-7182584873419195739,Low,E4 (17); E5 (3); E11 (3); E9 (3); E26 (3),E4
1,blk_-7182584873419195739,Low,E4 (17); E5 (3); E11 (3); E9 (3); E26 (3),E5
1,blk_-7182584873419195739,Low,E4 (17); E5 (3); E11 (3); E9 (3); E26 (3),E11
1,blk_-7182584873419195739,Low,E4 (17); E5 (3); E11 (3); E9 (3); E26 (3),E9
1,blk_-7182584873419195739,Low,E4 (17); E5 (3); E11 (3); E9 (3); E26 (3),E26


In [3]:
# Count distinct incidents containing each event within each severity

severity_event_summary = (
    events_exploded
    .groupby(["Severity", "EventId"])["BlockId"]
    .nunique()
    .reset_index(name="Incident_Count")
)

# Calculate total incidents within each severity
severity_totals = (
    incidents_df
    .groupby("Severity")["BlockId"]
    .nunique()
)

severity_event_summary["Severity_Total"] = (
    severity_event_summary["Severity"].map(severity_totals)
)

# Calculate percentage of incidents containing each event
severity_event_summary["Incident_Percentage"] = (
    severity_event_summary["Incident_Count"]
    / severity_event_summary["Severity_Total"] * 100
).round(2)

# Sort for easy inspection
severity_event_summary = severity_event_summary.sort_values(
    ["Severity", "Incident_Count"],
    ascending=[True, False]
).reset_index(drop=True)

print("Summary shape:", severity_event_summary.shape)
display(severity_event_summary.head(15))

Summary shape: (58, 5)


,Severity,EventId,Incident_Count,Severity_Total,Incident_Percentage
0,Critical,E5,1948,1948,100.00
1,Critical,E7,1907,1948,97.90
2,Critical,E22,1886,1948,96.82
3,Critical,E11,75,1948,3.85
4,Critical,E26,52,1948,2.67
5,Critical,E14,38,1948,1.95
6,Critical,E21,26,1948,1.33
7,Critical,E10,20,1948,1.03
8,Critical,E23,20,1948,1.03
9,Critical,E9,10,1948,0.51


In [4]:
# Save severity-wise event frequency for Power BI

output_path = Path("../outputs/hdfs_event_frequency_by_severity.csv")

severity_event_summary.to_csv(output_path, index=False)

print("File saved successfully!")
print("Saved location:", output_path.resolve())
print("Rows saved:", len(severity_event_summary))

# Validate the saved file
validation_df = pd.read_csv(output_path)

print("\nValidation:")
print("Shape:", validation_df.shape)
print("Missing values:", validation_df.isnull().sum().sum())

display(validation_df.head(10))

File saved successfully!
Saved location: D:\Srinath Folder\HDFS_Log_Anomaly_Detection_Project_Structure\HDFS_Log_Anomaly_Detection\outputs\hdfs_event_frequency_by_severity.csv
Rows saved: 58

Validation:
Shape: (58, 5)
Missing values: 0


,Severity,EventId,Incident_Count,Severity_Total,Incident_Percentage
0,Critical,E5,1948,1948,100.00
1,Critical,E7,1907,1948,97.90
2,Critical,E22,1886,1948,96.82
3,Critical,E11,75,1948,3.85
4,Critical,E26,52,1948,2.67
5,Critical,E14,38,1948,1.95
6,Critical,E21,26,1948,1.33
7,Critical,E10,20,1948,1.03
8,Critical,E23,20,1948,1.03
9,Critical,E9,10,1948,0.51


In [5]:
from pathlib import Path
import duckdb

# Locate the existing project database
db_path = Path("../database/hdfs_anomaly.duckdb")

print("Database exists:", db_path.exists())
print("Database path:", db_path.resolve())

if db_path.exists():
    con = duckdb.connect(str(db_path), read_only=True)

    tables = con.execute("SHOW TABLES").df()

    print("\nExisting DuckDB tables:")
    display(tables)

    con.close()
else:
    print("Database not found at the expected location.")

Database exists: True
Database path: D:\Srinath Folder\HDFS_Log_Anomaly_Detection_Project_Structure\HDFS_Log_Anomaly_Detection\database\hdfs_anomaly.duckdb

Existing DuckDB tables:


,name
0,anomaly_labels
1,event_matrix
2,event_traces


In [6]:
import duckdb

con = duckdb.connect(str(db_path), read_only=True)

for table_name in ["anomaly_labels", "event_matrix", "event_traces"]:
    print(f"\n{'='*50}")
    print(f"TABLE: {table_name}")
    print("="*50)

    schema = con.execute(
        f"DESCRIBE SELECT * FROM {table_name}"
    ).df()

    display(schema)

    count = con.execute(
        f"SELECT COUNT(*) FROM {table_name}"
    ).fetchone()[0]

    print("Total records:", count)

con.close()


TABLE: anomaly_labels


,column_name,column_type,null,key,default,extra
0,BlockId,VARCHAR,YES,None,None,None
1,Label,VARCHAR,YES,None,None,None


Total records: 575061

TABLE: event_matrix


,column_name,column_type,null,key,default,extra
0,BlockId,VARCHAR,YES,None,None,None
1,Label,VARCHAR,YES,None,None,None
2,Type,BIGINT,YES,None,None,None
3,E1,BIGINT,YES,None,None,None
4,E2,BIGINT,YES,None,None,None
5,E3,BIGINT,YES,None,None,None
6,E4,BIGINT,YES,None,None,None
7,E5,BIGINT,YES,None,None,None
8,E6,BIGINT,YES,None,None,None
9,E7,BIGINT,YES,None,None,None


Total records: 575061

TABLE: event_traces


,column_name,column_type,null,key,default,extra
0,BlockId,VARCHAR,YES,None,None,None
1,Label,VARCHAR,YES,None,None,None
2,Type,BIGINT,YES,None,None,None
3,Features,VARCHAR,YES,None,None,None
4,TimeInterval,VARCHAR,YES,None,None,None
5,Latency,BIGINT,YES,None,None,None


Total records: 575061


In [7]:
import duckdb

print("DuckDB version:", duckdb.__version__)

DuckDB version: 2.0.1


In [8]:
import sys
from pathlib import Path

# Add project root to Python path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.api.database import initialize_database, DB_PATH

# Initialize prediction history table
initialize_database()

print("Database initialized successfully!")
print("Database:", DB_PATH)

Database initialized successfully!
Database: D:\Srinath Folder\HDFS_Log_Anomaly_Detection_Project_Structure\HDFS_Log_Anomaly_Detection\database\hdfs_anomaly.duckdb


In [9]:
import duckdb

con = duckdb.connect(str(DB_PATH), read_only=True)

tables = con.execute("SHOW TABLES").df()
display(tables)

print("\nPrediction history schema:")
display(
    con.execute(
        "DESCRIBE prediction_history"
    ).df()
)

print(
    "\nExisting prediction records:",
    con.execute(
        "SELECT COUNT(*) FROM prediction_history"
    ).fetchone()[0]
)

con.close()

,name
0,anomaly_labels
1,event_matrix
2,event_traces
3,prediction_history



Prediction history schema:


,column_name,column_type,null,key,default,extra
0,prediction_id,VARCHAR,NO,PRI,None,None
1,predicted_at,TIMESTAMP,YES,NaN,None,None
2,event_ids,VARCHAR,YES,NaN,None,None
3,sequence_length,INTEGER,YES,NaN,None,None
4,reconstruction_score,DOUBLE,YES,NaN,None,None
5,threshold,DOUBLE,YES,NaN,None,None
6,prediction,VARCHAR,YES,NaN,None,None



Existing prediction records: 0


In [10]:
import duckdb
import pandas as pd

con = duckdb.connect(str(DB_PATH), read_only=True)

# Check total prediction records
count = con.execute("""
    SELECT COUNT(*) 
    FROM prediction_history
""").fetchone()[0]

print("Total stored predictions:", count)

# Retrieve the latest prediction
latest_prediction = con.execute("""
    SELECT *
    FROM prediction_history
    ORDER BY predicted_at DESC
    LIMIT 5
""").df()

display(latest_prediction)

con.close()

Total stored predictions: 1


,prediction_id,predicted_at,event_ids,sequence_length,reconstruction_score,threshold,prediction
0,be7b6063-ef8b-43ca-b518-c0e547c64d6d,2026-11-26 12:35:56.538056,"[5, 22, 5, 5, 11, 9, 11, 9, 11, 9, 26, 26, 26,...",22,0.68634,0.883776,NORMAL


In [11]:
import pandas as pd
import ast
from pathlib import Path

encoded_path = Path("../data/processed/encoded_sequences.csv")

encoded_df = pd.read_csv(encoded_path)

critical_block_id = "blk_8006271611835981128"

row = encoded_df.loc[
    encoded_df["BlockId"] == critical_block_id
].iloc[0]

critical_sequence = ast.literal_eval(
    row["EncodedFeatures"]
)

print("Block ID:", critical_block_id)
print("Sequence length:", len(critical_sequence))
print("Event sequence:", critical_sequence)

Block ID: blk_8006271611835981128
Sequence length: 49
Event sequence: (5, 5, 5, 22, 5, 11, 10, 14, 7, 8, 11, 15, 5, 11, 9, 26, 14, 7, 11, 10, 12, 14, 7, 5, 19, 14, 7, 11, 10, 14, 7, 8, 11, 25, 18, 5, 5, 6, 16, 6, 26, 26, 23, 23, 23, 21, 21, 21, 20)


In [12]:
import duckdb

con = duckdb.connect(str(DB_PATH), read_only=True)

# Prediction count by classification
summary = con.execute("""
    SELECT
        prediction,
        COUNT(*) AS total_predictions,
        ROUND(AVG(reconstruction_score), 4) AS avg_score
    FROM prediction_history
    GROUP BY prediction
    ORDER BY prediction
""").df()

display(summary)

# Inspect all stored predictions
records = con.execute("""
    SELECT
        prediction_id,
        sequence_length,
        reconstruction_score,
        prediction,
        predicted_at
    FROM prediction_history
    ORDER BY predicted_at
""").df()

display(records)

con.close()

,prediction,total_predictions,avg_score
0,ANOMALY,1,7.1888
1,NORMAL,1,0.6863


,prediction_id,sequence_length,reconstruction_score,prediction,predicted_at
0,be7b6063-ef8b-43ca-b518-c0e547c64d6d,22,0.686340,NORMAL,2026-11-26 12:35:56.538056
1,52c931b9-3110-49c4-9a87-219ba8d2f884,49,7.188761,ANOMALY,2026-11-26 12:39:01.983199


In [14]:
from pathlib import Path

database_file = Path("../src/api/database.py")

print("File exists:", database_file.exists())
print("File path:", database_file.resolve())

print("\nDoes the function exist in the saved file?")

source = database_file.read_text(encoding="utf-8")

print("get_dashboard_summary" in source)
print("\nLast 40 lines:")
print("\n".join(source.splitlines()[-40:]))

File exists: True
File path: D:\Srinath Folder\HDFS_Log_Anomaly_Detection_Project_Structure\HDFS_Log_Anomaly_Detection\src\api\database.py

Does the function exist in the saved file?
True

Last 40 lines:
        "prediction_id": prediction_id,
        "predicted_at": predicted_at.isoformat()
    }


def get_dashboard_summary():
    """Return KPI metrics from DuckDB prediction history."""

    with duckdb.connect(str(DB_PATH), read_only=True) as con:

        result = con.execute("""
            SELECT
                COUNT(*) AS total_predictions,

                COUNT(*) FILTER (
                    WHERE prediction = 'NORMAL'
                ) AS normal_predictions,

                COUNT(*) FILTER (
                    WHERE prediction = 'ANOMALY'
                ) AS anomaly_predictions,

                ROUND(
                    AVG(reconstruction_score), 4
                ) AS average_reconstruction_score,

                MAX(predicted_at) AS latest_prediction_at

            

In [15]:
import importlib
import src.api.database as database

importlib.reload(database)

summary = database.get_dashboard_summary()

print("Dashboard Summary:")
display(summary)

Dashboard Summary:


{'total_predictions': 2,
 'normal_predictions': 1,
 'anomaly_predictions': 1,
 'average_reconstruction_score': 3.9376,
 'latest_prediction_at': '2026-11-26T12:39:01.983199'}